# NB04 — Satellite prediction of Musanga cover fraction from UAV mosaics (V3)

**Objective:** predict Musanga canopy fraction from year-matched Sentinel-2 and annual Google Satellite Embeddings, train across forest sites and evaluate independent site transfer, blocked within-site extrapolation, and annotated-circle transfer at multiple spatial supports.

**Sources:** new frozen NB03 native-RGB M2 full probability maps; V2/08 UAV-acquisition years; V2 native Sentinel-2 10 m projection registry and satellite feature definitions. Source UAV probabilities are *model-derived pseudo-labels*, not independent ground truth. The manually annotated 30 V3 exhaustive circles are an additional diagnostic, but the final M2 that produced pseudo-labels was trained on these circles, so independence is from satellite-model training only, **not** from UAV-model training. Never claim circle comparisons validate UAV probabilities independently.

**Acquisition years from V2:** Baego 2023, Yoko 2020, Yangambi 2024, MOS 2024, optional Luki 2023. **Assumption to verify:** both MOS 3–4 and MOS 5–6 were acquired in the same year as MOS 1–2. Do not interpret within-year temporality as established. Year−1/year/year+1 predictors are retrospective and may contain post-UAV imagery.

**Workflow:** 00 manifest and years → 01 exact S2-aligned grids → 02 begin year-specific satellite downloads (independent of NB03) → 03 aggregate full-UAV probability and thresholded cover with direct source-to-grid area support and coverage masks at 10/20/30/50/75/100 m → 04 build matched predictor/target samples at fixed scales → 05 leave-one-site-out and spatial-block model comparison → 06 independent-annotation-circle diagnostics → 07 diagnostic figures, audit, and best-model refit for satellite maps. Blocks requiring NB03 products stop with an actionable missing-file list; you can download satellite inputs while NB03 runs.

**Analysis distinction:** mean UAV Musanga probability (expected cover) and area fraction of pixels with probability≥0.5 (threshold cover) are separate responses, always with UAV-valid-area fraction. Evaluate both, never label either as field-measured cover. Exclude cells with <90% UAV coverage and near-zero S2 feature support; retain coverage diagnostics. Explicit physical areas and aligned grids are essential, especially 75 m which must be calculated directly (not by stacking 10 m rasters).

**Evaluation contract:** all comparisons use matched geographic cells per scale/family (intersection support); report per-site errors and whole-mosaic maps, and the 30 circle-level observed cover fractions separately. Never randomly split neighboring 10 m pixels: primary leave-one-site-out (Baego/MOS/Yangambi/Yoko; all three MOS mosaics belong to one group); secondary blocked ~500 m grid with ≥100 m spatial buffer and no leakage among overlapping scales. Circle reference polygons and buffers are **excluded from satellite model training** when using pseudo-targets; for site-LOSO, complete held-out site is excluded. Rank models using outer-fold predictions only.

**Start here:** Run setup, grid registry and download cells first. Google Earth Engine authentication must be performed interactively in your Colab account; this notebook cannot authenticate or launch cloud jobs by itself. Downloads are configurable, checkpointed, and never overwrite existing verified raster files.


In [ ]:
%pip -q install earthengine-api geemap geedim rasterio geopandas shapely scikit-learn pandas matplotlib scipy tqdm
import json,os,math,hashlib,shutil,time,warnings
from pathlib import Path
import numpy as np,pandas as pd,rasterio
from rasterio.windows import Window
from rasterio.warp import reproject,Resampling,transform_bounds
from affine import Affine
from IPython.display import display
from google.colab import drive
if not Path("/content/drive/MyDrive").exists():drive.mount("/content/drive")
ROOT=next((p for p in [Path("/content/drive/MyDrive/Colab Notebooks/Musanga/Final"),
                         Path("/content/drive/MyDrive/Musanga/Final")] if p.is_dir()),None)
assert ROOT is not None
LEGACY=ROOT.parent/"KVO_work/KVO_08_satellite_fraction"
LEGACY_GRIDS=LEGACY/"validation/09_s2_canonical_10m_grids.json"
MANIFEST=ROOT/"Outputs/V3/00_input_audit/00_input_manifest.json"
assert MANIFEST.exists(),MANIFEST
A=json.loads(MANIFEST.read_text())
assert A.get("schema_version")=="v3_input_audit_5_nadine_circles"
SOURCES={m["file"]:dict(site=m["site"],path=Path(m["path"])) for m in A["mosaics"]}
assert len(SOURCES)>=6 and all(q["path"].exists() for q in SOURCES.values())
YEARS={"Baego":2023,"Yoko":2020,"Yangambi":2024,"MOS":2024,"Luki":2023}
PROVENANCE={"Baego":"V2/08","Yoko":"V2/08","Yangambi":"V2/08",
            "MOS":"V2/08 for MOS_1_2; extrapolated provisionally to MOS_3_4/MOS_5_6",
            "Luki":"V2/08; exact acquisition date unresolved"}
INCLUDE_LUKI=False
if INCLUDE_LUKI:
    p=ROOT.parent/"data/Geo_Ref/Luki.tif"
    assert p.exists()
    SOURCES["Luki.tif"]=dict(site="Luki",path=p)
SITES=sorted(set(x["site"] for x in SOURCES.values()))
OUT=ROOT/"Outputs/V3/04_satellite_Musanga"
GRID_DIR=OUT/"grid";FEAT_DIR=OUT/"features";AGG_DIR=OUT/"uav_aggregated"
MODEL_DIR=OUT/"models";QA_DIR=OUT/"qa"
for p in (GRID_DIR,FEAT_DIR,AGG_DIR,MODEL_DIR,QA_DIR):p.mkdir(parents=True,exist_ok=True)
MODE="final_all30"   # NB03 model, must match native RGB probability maps
NB03=ROOT/("Outputs/V3/03_M2_full_mosaic_"+MODE)/"probability_maps"
THRESHOLD=0.5
MIN_UAV_COVERAGE=.90
MIN_FEATURE_VALID=.90
SCALES=[10,20,30,50,75,100]
assert set(SCALES)=={10,20,30,50,75,100}
assert all(x["site"] in YEARS for x in SOURCES.values())
reg=[]
for m,x in SOURCES.items():
    with rasterio.open(x["path"]) as d:
        assert d.crs is not None
        reg.append(dict(mosaic=m,site=x["site"],year=YEARS[x["site"]],
                        crs=str(d.crs),width=d.width,height=d.height,
                        raster_GB=round(x["path"].stat().st_size/1e9,3),
                        year_basis=PROVENANCE[x["site"]]))
display(pd.DataFrame(reg))
(OUT/"04_input_registry.json").write_text(json.dumps(
    dict(years=YEARS,provenance=PROVENANCE,sources={k:dict(site=v["site"],path=str(v["path"]))
    for k,v in SOURCES.items()},uav_model=MODE),indent=2))
print("NB03 files may still be running. Satellite download blocks do not require NB03.")


## 01 — Canonical 10 m satellite grid and independent 75 m aggregation grids

Use V2's **actual native S2 B2 tile CRS/origin** wherever available. For MOS multi-mosaic analysis, extend a single MOS S2 grid to the union of the mosaics and retain MOS as one cross-validation site. Coarser grids inherit the same UTM origin; 75 m is independent and never formed by averaging prefiltered 10 m pixels.


In [ ]:
assert LEGACY_GRIDS.exists(),(
    "V2/08 native Sentinel-2 grid registry missing; build it in V2/08 Block 02 first",LEGACY_GRIDS)
old=json.loads(LEGACY_GRIDS.read_text())
SITE_V2={"Baego":"Baego","Yangambi":"Yangambi","Yoko":"Yoko",
         "MOS":"MOS_1_2","Luki":"Luki"}
GRIDS={}
for site in SITES:
    key=SITE_V2[site]
    assert key in old,("Missing V2 grid",site,key)
    g=old[key]
    assert int(g["uav_year"])==YEARS[site],(site,g["uav_year"])
    crs=g["uav_crs"]
    # ORIGINAL native scene grid: universal origin for *all* output scales,
    # independent of each site's mosaic footprint.
    native=Affine(*g["native_scene_transform"])
    assert abs(native.a-10)<1e-6 and abs(native.e+10)<1e-6
    assert abs(native.b)<1e-9 and abs(native.d)<1e-9
    rasters=[q["path"] for q in SOURCES.values() if q["site"]==site]
    bounds=[]
    for p in rasters:
        with rasterio.open(p) as ds:
            assert str(ds.crs)==crs,("Mosaic and canonical S2 CRS differ",site,p,ds.crs,crs)
            bounds.append(ds.bounds)
    west=min(b.left for b in bounds);south=min(b.bottom for b in bounds)
    east=max(b.right for b in bounds);north=max(b.top for b in bounds)
    scale=10
    x0=native.c+math.floor((west-native.c)/scale)*scale
    y0=native.f-math.floor((native.f-north)/scale)*scale
    nx=math.ceil((east-x0)/scale);ny=math.ceil((y0-south)/scale)
    GRIDS[site]=dict(site=site,year=YEARS[site],crs=crs,
        native_scene_transform=list(native)[:6],transform=[10,0,x0,0,-10,y0],
        width=nx,height=ny,bounds=[west,south,east,north],
        mosaics=[k for k,q in SOURCES.items() if q["site"]==site],
        scale=10,s2_native_tile=g["s2_native_tile"])
    print(site,"S2 native CRS",crs,"grid",nx,ny,"year",YEARS[site],
          "tile",g["s2_native_tile"])
(OUT/"04_canonical_site_grids.json").write_text(json.dumps(GRIDS,indent=2))

def scaled_grid(site,scale):
    """Exact nested physical UTM grid; does not assume scale divides 10."""
    g=GRIDS[site];native=Affine(*g["native_scene_transform"])
    west,south,east,north=g["bounds"]
    x0=native.c+math.floor((west-native.c)/scale)*scale
    y0=native.f-math.floor((native.f-north)/scale)*scale
    nx=math.ceil((east-x0)/scale);ny=math.ceil((y0-south)/scale)
    return dict(site=site,scale_m=scale,crs=g["crs"],
                transform=[scale,0,x0,0,-scale,y0],width=nx,height=ny)
for scale in SCALES:
    assert all(scaled_grid(s,scale)["width"]>0 for s in SITES)
print("Canonical grids ready. Features can now be downloaded independently of NB03.")
